# 📰 Task 1: News Topic Classifier Using BERT

**Objective:** Fine-tune `bert-base-uncased` on the AG News Dataset to classify news headlines into 4 categories:
- 0 → World
- 1 → Sports
- 2 → Business
- 3 → Sci/Tech

---

## 📦 Step 1 — Install Dependencies

In [1]:
# DO NOT reinstall torch or torchvision — Colab's preinstalled pair is already
# matched to its GPU/CUDA build. Reinstalling either one breaks the ABI match
# between them (RuntimeError: operator torchvision::nms does not exist).
# We only pin transformers/accelerate to versions compatible with Colab's torch.
!pip install -U datasets huggingface_hub fsspec "transformers==4.44.2" "accelerate==0.34.2" scikit-learn gradio -q
# IMPORTANT: after this finishes, go to Runtime -> Restart session,
# then re-run ALL cells from the top in this exact freshly-restarted session.

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.4/203.4 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.3/32.3 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 40.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently

## 📥 Step 2 — Load AG News Dataset

In [2]:
from datasets import load_dataset

dataset = load_dataset('fancyzhx/ag_news')
print(dataset)
print('\nSample entry:')
print(dataset['train'][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

Sample entry:
{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}


## 🔍 Step 3 — Explore the Dataset

In [3]:
import pandas as pd

LABEL_NAMES = ['World', 'Sports', 'Business', 'Sci/Tech']

# Convert to pandas for easy exploration
train_df = pd.DataFrame(dataset['train'])
train_df['category'] = train_df['label'].map(lambda x: LABEL_NAMES[x])

print('Dataset shape:', train_df.shape)
print('\nClass distribution:')
print(train_df['category'].value_counts())

train_df[['text', 'category']].head(10)

Dataset shape: (120000, 3)

Class distribution:
category
Business    30000
Sci/Tech    30000
Sports      30000
World       30000
Name: count, dtype: int64


,text,category
0,Wall St. Bears Claw Back Into the Black (Reute...,Business
1,Carlyle Looks Toward Commercial Aerospace (Reu...,Business
2,Oil and Economy Cloud Stocks' Outlook (Reuters...,Business
3,Iraq Halts Oil Exports from Main Southern Pipe...,Business
4,"Oil prices soar to all-time record, posing new...",Business
5,"Stocks End Up, But Near Year Lows (Reuters) Re...",Business
6,Money Funds Fell in Latest Week (AP) AP - Asse...,Business
7,Fed minutes show dissent over inflation (USATO...,Business
8,Safety Net (Forbes.com) Forbes.com - After ear...,Business
9,Wall St. Bears Claw Back Into the Black NEW Y...,Business


## ✂️ Step 4 — Tokenize with BERT Tokenizer

In [4]:
from transformers import BertTokenizer

MODEL_NAME = 'bert-base-uncased'
MAX_LEN = 128

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

# Demo tokenization on one sample
sample = dataset['train'][0]['text']
tokens = tokenizer(sample, max_length=MAX_LEN, truncation=True, padding='max_length')
print('Text     :', sample[:100], '...')
print('Input IDs:', tokens['input_ids'][:20], '...')
print('Tokens   :', tokenizer.convert_ids_to_tokens(tokens['input_ids'][:20]))

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Text     : Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\b ...
Input IDs: [101, 2813, 2358, 1012, 6468, 15020, 2067, 2046, 1996, 2304, 1006, 26665, 1007, 26665, 1011, 2460, 1011, 19041, 1010, 2813] ...
Tokens   : ['[CLS]', 'wall', 'st', '.', 'bears', 'claw', 'back', 'into', 'the', 'black', '(', 'reuters', ')', 'reuters', '-', 'short', '-', 'sellers', ',', 'wall']


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [5]:
# Use a subset for faster training in notebook (increase for full run)
train_dataset = dataset['train'].shuffle(seed=42).select(range(8000))
test_dataset  = dataset['test'].shuffle(seed=42).select(range(2000))

def tokenize(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=MAX_LEN)

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset  = test_dataset.map(tokenize, batched=True)

train_dataset = train_dataset.rename_column('label', 'labels')
test_dataset  = test_dataset.rename_column('label', 'labels')

train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
test_dataset.set_format('torch',  columns=['input_ids', 'attention_mask', 'labels'])

print('Train size:', len(train_dataset))
print('Test size :', len(test_dataset))

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Train size: 8000
Test size : 2000


## 🤖 Step 5 — Load BERT & Fine-tune

In [6]:
import numpy as np
from transformers import BertForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=4)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1  = f1_score(labels, predictions, average='weighted')
    return {'accuracy': acc, 'f1': f1}

training_args = TrainingArguments(
    output_dir='./bert_ag_news',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_steps=50,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.312300,0.306929,0.905500,0.905284
2,0.268700,0.298849,0.916000,0.916100
3,0.075800,0.341591,0.918500,0.918539


TrainOutput(global_step=1500, training_loss=0.229492715994517, metrics={'train_runtime': 666.6467, 'train_samples_per_second': 36.001, 'train_steps_per_second': 2.25, 'total_flos': 1578694680576000.0, 'train_loss': 0.229492715994517, 'epoch': 3.0})

## 📊 Step 6 — Evaluate Model

In [7]:
results = trainer.evaluate()
print(f"\n=== FINAL EVALUATION ===")
print(f"Accuracy : {results['eval_accuracy']:.4f}")
print(f"F1-Score : {results['eval_f1']:.4f}")


=== FINAL EVALUATION ===
Accuracy : 0.9185
F1-Score : 0.9185


In [8]:
# Detailed classification report
from sklearn.metrics import classification_report
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()

all_preds, all_labels = [], []

from torch.utils.data import DataLoader
loader = DataLoader(test_dataset, batch_size=32)

with torch.no_grad():
    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels']
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds   = torch.argmax(outputs.logits, dim=-1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=LABEL_NAMES))

              precision    recall  f1-score   support

       World       0.93      0.93      0.93       497
      Sports       0.98      0.98      0.98       483
    Business       0.90      0.87      0.88       522
    Sci/Tech       0.87      0.91      0.89       498

    accuracy                           0.92      2000
   macro avg       0.92      0.92      0.92      2000
weighted avg       0.92      0.92      0.92      2000



## 💾 Step 7 — Save Model

In [9]:
model.save_pretrained('./bert_ag_news')
tokenizer.save_pretrained('./bert_ag_news')
print('Model and tokenizer saved to ./bert_ag_news/')

Model and tokenizer saved to ./bert_ag_news/


## 🚀 Step 8 — Gradio Live Demo

In [10]:
import gradio as gr
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()

def predict(text: str):
    if not text.strip():
        return 'Please enter a headline.'
    inputs = tokenizer(
        text, return_tensors='pt',
        padding=True, truncation=True, max_length=128
    ).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()
    return {LABEL_NAMES[i]: float(probs[i]) for i in range(4)}

gr.Interface(
    fn=predict,
    inputs=gr.Textbox(lines=2, placeholder='Enter a news headline...', label='Headline'),
    outputs=gr.Label(num_top_classes=4, label='Category Probabilities'),
    title='📰 News Topic Classifier (BERT)',
    description='Fine-tuned bert-base-uncased on AG News | World · Sports · Business · Sci/Tech',
    examples=[
        ['NASA launches new Mars rover to search for ancient life'],
        ['Stock markets surge as inflation drops to 2-year low'],
        ['Real Madrid wins Champions League final in extra time'],
        ['New AI chip from NVIDIA claims 10x speed over H100'],
    ]
).launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://42a38806454dc5159a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [11]:
model.save_pretrained('/content/drive/MyDrive/bert_ag_news')  # if Drive is mounted
tokenizer.save_pretrained('/content/drive/MyDrive/bert_ag_news')

('/content/drive/MyDrive/bert_ag_news/tokenizer_config.json',
 '/content/drive/MyDrive/bert_ag_news/special_tokens_map.json',
 '/content/drive/MyDrive/bert_ag_news/vocab.txt',
 '/content/drive/MyDrive/bert_ag_news/added_tokens.json')